# <font color="#418FDE" size="6.5" uppercase>**Logistic LDA QDA**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Fit logistic regression models for binary and multiclass classification. 
- Configure penalties, solvers, and class weights for practical classification problems. 
- Compare Logistic Regression, LDA, and QDA on suitable datasets. 


## **1. Logistic Models**

### **1.1. Binary Logistic Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_01_01.jpg?v=1783961976" width="250">



>* Models probabilities for two-class outcomes
>* Classifies using an adjustable probability threshold

>* Coefficients show feature effects on class likelihood
>* Probability effects vary across the curve

>* Learn coefficients that define class boundaries
>* Use probabilities for simple, interpretable classification



In [ ]:
#@title Python Code - Binary Logistic Regression

# Binary logistic regression predicts two possible classes.
# This example implements training using NumPy only.
# We visualize probabilities and the decision boundary.

import numpy as np
import matplotlib.pyplot as plt

# Set a seed for reproducible synthetic data.
rng = np.random.default_rng(7)

# Create two small feature groups in centimeters.
class_zero = rng.normal(loc=[155, 55], scale=[6, 5], size=(40, 2))
class_one = rng.normal(loc=[172, 72], scale=[7, 6], size=(40, 2))

# Stack features and binary labels together.
X_raw = np.vstack([class_zero, class_one])
y = np.r_[np.zeros(40), np.ones(40)]

# Validate the tiny dataset before training.
assert X_raw.shape == (80, 2)
assert y.shape == (80,)

# Standardize features for stable gradient descent.
means = X_raw.mean(axis=0)
scales = X_raw.std(axis=0)
X_scaled = (X_raw - means) / scales

# Add an intercept column to the design matrix.
ones = np.ones((X_scaled.shape[0], 1))
X = np.hstack([ones, X_scaled])

# Define the sigmoid probability transformation.
def sigmoid(values):
    clipped = np.clip(values, -30, 30)
    return 1 / (1 + np.exp(-clipped))

# Train logistic regression with simple gradient descent.
weights = np.zeros(X.shape[1])
learning_rate = 0.15
steps = 900

# Update coefficients using average logistic loss gradient.
for step in range(steps):
    probabilities = sigmoid(X @ weights)
    gradient = X.T @ (probabilities - y) / y.size
    weights -= learning_rate * gradient

# Convert probabilities into class predictions.
probabilities = sigmoid(X @ weights)
predictions = (probabilities >= 0.5).astype(int)
accuracy = (predictions == y).mean()

# Print a compact summary for learners.
print(f"Training accuracy: {accuracy:.2f}")
print(f"Intercept: {weights[0]:.2f}")
print(f"Height coefficient: {weights[1]:.2f}")
print(f"Weight coefficient: {weights[2]:.2f}")

# Build a small grid for the probability surface.
x_min, x_max = X_raw[:, 0].min() - 5, X_raw[:, 0].max() + 5
y_min, y_max = X_raw[:, 1].min() - 5, X_raw[:, 1].max() + 5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 120), np.linspace(y_min, y_max, 120))

# Standardize grid points using training statistics.
grid_raw = np.c_[xx.ravel(), yy.ravel()]
grid_scaled = (grid_raw - means) / scales
grid_X = np.c_[np.ones(grid_scaled.shape[0]), grid_scaled]

# Predict probabilities across the grid.
grid_prob = sigmoid(grid_X @ weights).reshape(xx.shape)

# Plot data, probabilities, and the boundary.
plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, grid_prob, levels=20, cmap="RdBu", alpha=0.35)
plt.contour(xx, yy, grid_prob, levels=[0.5], colors="black", linewidths=2)
plt.scatter(class_zero[:, 0], class_zero[:, 1], label="Class 0", edgecolor="black")
plt.scatter(class_one[:, 0], class_one[:, 1], label="Class 1", edgecolor="black")

# Label the axes and show the plot.
plt.xlabel("Height in centimeters")
plt.ylabel("Weight in kilograms")
plt.title("Binary Logistic Regression Probability Boundary")
plt.legend()
plt.show()



### **1.2. Multiclass Logistic**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_01_02.jpg?v=1783961978" width="250">



>* Classifies observations into multiple possible categories
>* Outputs probabilities showing confidence across classes

>* One-versus-rest builds separate binary classifiers
>* Multinomial models coordinate probabilities across classes

>* Treat classes as distinct, not ordered
>* Check class balance and probability confidence



### **1.3. Penalty Choices**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_01_03.jpg?v=1783961980" width="250">



>* Penalties limit overly large model coefficients
>* They improve stability and future predictions

>* Penalties shrink or remove model coefficients
>* Multiclass penalties reduce overreaction to rare signals

>* Tune penalty strength using validation performance
>* Standardize features before penalized logistic regression



## **2. Solver Choices**

### **2.1. Solver compatibility**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_02_01.jpg?v=1783961973" width="250">



>* Match solvers to data and penalties
>* Avoid unsupported or inefficient model choices

>* Match solvers to regularization penalties
>* Use L1 for feature selection

>* Match solvers to multiclass strategy
>* Check scaling, convergence, and model stability



### **2.2. Balanced Class Weights**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_02_02.jpg?v=1783961966" width="250">



>* Class imbalance can hide poor minority performance
>* Balanced weights emphasize underrepresented classes

>* Useful when rare classes matter most
>* Weights reflect real error consequences

>* Weighting trades recall against false alarms
>* Validate with metrics and domain consequences



### **2.3. Sample Weighting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_02_03.jpg?v=1783961957" width="250">



>* Higher weights give examples more influence
>* Emphasizes importance without changing data labels

>* Weight individual examples, not just classes
>* Reflect costs, surveys, and real priorities

>* Choose weights with clear practical purpose
>* Evaluate using matching weighted goals



## **3. Discriminant Analysis**

### **3.1. LDA Classifier**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_03_01.jpg?v=1783961950" width="250">



>* Models each class by its feature patterns
>* Classifies using centers, variation, and separation

>* Shared covariance creates linear class boundaries
>* Stable for smaller datasets than QDA

>* Logistic regression and LDA use different assumptions
>* Validate LDA against alternatives and data fit



### **3.2. QDA Classifier**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_03_02.jpg?v=1783961952" width="250">



>* QDA allows class-specific covariance patterns
>* Models differing spreads, orientations, and correlations

>* QDA handles curved class boundaries
>* Useful when class spreads differ

>* QDA needs enough data to avoid overfitting
>* Use QDA for truly different class shapes



### **3.3. Covariance Shrinkage**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_03_03.jpg?v=1783961954" width="250">



>* Stabilizes covariance estimates in difficult datasets
>* Reduces overfitting for better unseen performance

>* Use cautious covariance estimates for uncertain data
>* Shrinkage smooths LDA and QDA boundaries

>* Shrinkage regularizes LDA and QDA covariance estimates
>* Choose models using validation and data structure



In [ ]:
#@title Python Code - Covariance Shrinkage

# Compare discriminant models with shrinkage.
# Small synthetic data keeps runtime low.
# Results show validation accuracy differences.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for reproducibility.
rng = np.random.default_rng(42)

# Create correlated two dimensional class data.
mean_zero = np.array([0.0, 0.0])
mean_one = np.array([1.2, 1.0])

cov_zero = np.array([[1.0, 0.85], [0.85, 1.0]])
cov_one = np.array([[1.2, -0.75], [-0.75, 1.0]])

# Use few samples to make covariance unstable.
X0 = rng.multivariate_normal(mean_zero, cov_zero, 28)
X1 = rng.multivariate_normal(mean_one, cov_one, 28)

X = np.vstack([X0, X1])
y = np.array([0] * 28 + [1] * 28)

# Shuffle before splitting into train and test.
order = rng.permutation(len(y))
X = X[order]
y = y[order]

split = 36
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Standardize using training statistics only.
mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0) + 1e-9

X_train = (X_train - mu) / sigma
X_test = (X_test - mu) / sigma

# Add intercept column for logistic regression.
Xb = np.c_[np.ones(len(X_train)), X_train]
weights = np.zeros(Xb.shape[1])

# Fit simple logistic regression with gradient descent.
for step in range(900):
    scores = Xb @ weights
    probs = 1 / (1 + np.exp(-scores))
    grad = Xb.T @ (probs - y_train) / len(y_train)
    weights -= 0.25 * grad

# Define covariance shrinkage helper function.
def shrink_cov(cov, alpha):
    target = np.eye(cov.shape[0]) * np.trace(cov) / cov.shape[0]
    return (1 - alpha) * cov + alpha * target

# Define Gaussian log density helper function.
def log_gaussian(Xdata, mean, cov):
    inv_cov = np.linalg.pinv(cov)
    sign, logdet = np.linalg.slogdet(cov)
    diff = Xdata - mean
    quad = np.sum((diff @ inv_cov) * diff, axis=1)
    return -0.5 * (quad + logdet)

# Define LDA prediction with optional shrinkage.
def predict_lda(Xdata, alpha):
    means = [X_train[y_train == k].mean(axis=0) for k in [0, 1]]
    pooled = np.cov(X_train.T, bias=False)
    cov = shrink_cov(pooled, alpha) + 1e-6 * np.eye(2)
    logs = [log_gaussian(Xdata, means[k], cov) for k in [0, 1]]
    return np.argmax(np.vstack(logs).T, axis=1)

# Define QDA prediction with optional shrinkage.
def predict_qda(Xdata, alpha):
    logs = []
    for k in [0, 1]:
        Xk = X_train[y_train == k]
        cov = np.cov(Xk.T, bias=False)
        cov = shrink_cov(cov, alpha) + 1e-6 * np.eye(2)
        logs.append(log_gaussian(Xdata, Xk.mean(axis=0), cov))
    return np.argmax(np.vstack(logs).T, axis=1)

# Evaluate all models on held out data.
logit_pred = ((np.c_[np.ones(len(X_test)), X_test] @ weights) > 0).astype(int)
lda_plain = predict_lda(X_test, 0.0)
lda_shrink = predict_lda(X_test, 0.5)
qda_plain = predict_qda(X_test, 0.0)
qda_shrink = predict_qda(X_test, 0.5)

# Store compact accuracy results for printing.
results = {
    "Logistic": np.mean(logit_pred == y_test),
    "LDA": np.mean(lda_plain == y_test),
    "Shrink LDA": np.mean(lda_shrink == y_test),
    "QDA": np.mean(qda_plain == y_test),
    "Shrink QDA": np.mean(qda_shrink == y_test),
}

# Print only a few teaching lines.
print("Covariance shrinkage pulls noisy covariance toward a stable target.")
for name, acc in results.items():
    print(f"{name:10s} test accuracy: {acc:.2f}")

# Plot the compact comparison as one figure.
plt.figure(figsize=(7, 4))
plt.bar(results.keys(), results.values(), color="steelblue")
plt.ylim(0, 1)
plt.ylabel("Test accuracy")
plt.title("Logistic, LDA, QDA, and covariance shrinkage")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Logistic LDA QDA**</font>


In this lecture, you learned to:
- Fit logistic regression models for binary and multiclass classification. 
- Configure penalties, solvers, and class weights for practical classification problems. 
- Compare Logistic Regression, LDA, and QDA on suitable datasets. 

In the next Lecture (Lecture B), we will go over 'Stochastic Gradient'